In [1]:
# import libraries and paths
import pandas as pd
from pathlib import Path

AUDIT = Path(r"C:\Users\Admin\Endometriosis\endo_project\audit")
SPLITS = Path(r"C:\Users\Admin\Endometriosis\endo_project\data\splits")
SPLITS.mkdir(parents=True, exist_ok=True)

df_files = pd.read_csv(AUDIT / "file_audit.csv")
df_files["rater"] = df_files["rater"].fillna("single")

def summarise_subject(g):
    seqs = set(g[g["modality"].isin(["T1", "T2", "T1FS", "T2FS"])]["modality"])
    return pd.Series({
        "centre":          g["centre"].iloc[0],
        "sequences":       sorted(seqs),
        "n_sequences":     len(seqs),
        "has_T1":          "T1"   in seqs,
        "has_T2":          "T2"   in seqs,
        "has_T1FS":        "T1FS" in seqs,
        "has_T2FS":        "T2FS" in seqs,
        "has_em":          (g["modality"] == "em").any(),
        "has_ov":          (g["modality"] == "ov").any(),
        "has_ut":          (g["modality"] == "ut").any(),
        "n_em_files":      int((g["modality"] == "em").sum()),
        "n_ov_files":      int((g["modality"] == "ov").sum()),
        "n_ut_files":      int((g["modality"] == "ut").sum()),
        "n_em_raters":     g[g["modality"] == "em"]["rater"].nunique(),
        "n_ov_raters":     g[g["modality"] == "ov"]["rater"].nunique(),
        "n_ut_raters":     g[g["modality"] == "ut"]["rater"].nunique(),
    })

df_subjects = (
    df_files.groupby("subject")
            .apply(summarise_subject)
            .reset_index()
)

print("Total subjects:", len(df_subjects))
print()
print("Per centre × has_em:")
print(df_subjects.groupby(["centre", "has_em"]).size())
print()
print("Per centre × has_ov:")
print(df_subjects.groupby(["centre", "has_ov"]).size())
print()
print("Sequence count distribution:")
print(df_subjects["n_sequences"].value_counts().sort_index())

df_subjects.head()

Total subjects: 124

Per centre × has_em:
centre   has_em
D1_MHS   False     11
         True      40
D2_TCPW  False     62
         True      11
dtype: int64

Per centre × has_ov:
centre   has_ov
D1_MHS   False      8
         True      43
D2_TCPW  False     15
         True      58
dtype: int64

Sequence count distribution:
n_sequences
0     3
1    13
2    38
3    34
4    36
Name: count, dtype: int64


,subject,centre,sequences,n_sequences,has_T1,has_T2,has_T1FS,has_T2FS,has_em,has_ov,has_ut,n_em_files,n_ov_files,n_ut_files,n_em_raters,n_ov_raters,n_ut_raters
0,D1-000,D1_MHS,"[T1FS, T2]",2,False,True,True,False,True,True,True,3,1,3,3,1,3
1,D1-001,D1_MHS,"[T1FS, T2]",2,False,True,True,False,True,True,True,3,3,3,3,3,3
2,D1-002,D1_MHS,"[T1FS, T2]",2,False,True,True,False,False,True,True,0,3,3,0,3,3
3,D1-003,D1_MHS,"[T1FS, T2]",2,False,True,True,False,True,True,True,3,2,3,3,2,3
4,D1-004,D1_MHS,[T2],1,False,True,False,False,True,True,True,1,3,3,1,3,3


In [2]:
# matched subsets for RQ2(Comparing sequences)
# All four sequences
all4 = df_subjects[df_subjects["sequences"].apply(
    lambda s: isinstance(s, list) and set(s) == {"T1", "T1FS", "T2", "T2FS"}
)]

# T1FS + T2 (largest 2-sequence subset)
t1fs_t2 = df_subjects[df_subjects["sequences"].apply(
    lambda s: isinstance(s, list) and {"T1FS", "T2"}.issubset(set(s))
)]

# T1 + T1FS + T2FS (missing T2)
t1_t1fs_t2fs = df_subjects[df_subjects["sequences"].apply(
    lambda s: isinstance(s, list) and set(s) == {"T1", "T1FS", "T2FS"}
)]

print("=== Subset: all 4 sequences ===")
print("N =", len(all4))
print(all4.groupby(["centre", "has_em"]).size())
print()

print("=== Subset: T1FS + T2 ===")
print("N =", len(t1fs_t2))
print(t1fs_t2.groupby(["centre", "has_em"]).size())
print()

print("=== Subset: T1 + T1FS + T2FS ===")
print("N =", len(t1_t1fs_t2fs))
print(t1_t1fs_t2fs.groupby(["centre", "has_em"]).size())

all4.to_csv(AUDIT / "subset_all4seq.csv", index=False)
t1fs_t2.to_csv(AUDIT / "subset_t1fs_t2.csv", index=False)
t1_t1fs_t2fs.to_csv(AUDIT / "subset_t1_t1fs_t2fs.csv", index=False)

=== Subset: all 4 sequences ===
N = 36
centre   has_em
D2_TCPW  False     29
         True       7
dtype: int64

=== Subset: T1FS + T2 ===
N = 75
centre   has_em
D1_MHS   False      6
         True      31
D2_TCPW  False     31
         True       7
dtype: int64

=== Subset: T1 + T1FS + T2FS ===
N = 23
centre   has_em
D2_TCPW  False     21
         True       2
dtype: int64


In [3]:
# save
df_subjects.to_csv(AUDIT / "subject_level.csv", index=False)
print("Saved:", AUDIT / "subject_level.csv")
print("Rows:", len(df_subjects))

Saved: C:\Users\Admin\Endometriosis\endo_project\audit\subject_level.csv
Rows: 124


In [ ]:
# 3 patients had zero sequence (who were they?)

AUDIT = Path(r"C:\Users\Admin\Endometriosis\endo_project\audit")
df_subjects = pd.read_csv(AUDIT / "subject_level.csv")
df_files = pd.read_csv(AUDIT / "file_audit.csv")

zero_seq = df_subjects[df_subjects["n_sequences"] == 0]["subject"].tolist()
print("Zero-sequence subjects:", zero_seq)
print()
for s in zero_seq:
    print(f"--- {s} ---")
    print(df_files[df_files["subject"] == s][["modality","rater","file"]].to_string(index=False))
    print()

Zero-sequence subjects: ['D1-031', 'D2-068', 'D2-069']

--- D1-031 ---
modality  rater                file
      em    3.0 D1-031_em_r3.nii.gz
      ov    3.0 D1-031_ov_r3.nii.gz
      ut    3.0 D1-031_ut_r3.nii.gz

--- D2-068 ---
modality  rater             file
      ov    NaN D2-068_ov.nii.gz
      ut    NaN D2-068_ut.nii.gz

--- D2-069 ---
modality  rater             file
      ut    NaN D2-069_ut.nii.gz

